In [3]:
!pip -q install spacy scikit-learn
!python -m spacy download en_core_web_sm

import spacy
import numpy as np

from itertools import combinations
from collections import Counter
from sklearn.svm import SVC


# LOAD SPACY NLP MODEL

nlp = spacy.load("en_core_web_sm")

print("SpaCy model loaded successfully!")


# EXTRACT DEPENDENCY PATH BETWEEN TWO ENTITIES

def get_dependency_path(ent1, ent2):

    token1 = ent1.root
    token2 = ent2.root

    ancestors1 = [token1] + list(token1.ancestors)
    ancestors2 = set([token2] + list(token2.ancestors))

    common = next(
        (token for token in ancestors1 if token in ancestors2),
        None
    )

    if common is None:
        return []

    def get_lemma(token):
        return (token.lemma_ or token.text).lower()

    path = [get_lemma(token1)]

    token = token1

    while token != common:

        path.append("UP:" + token.dep_)

        token = token.head

        path.append(get_lemma(token))

    down_path = []

    token = token2

    while token != common:

        down_path.append(
            (token.dep_, get_lemma(token))
        )

        token = token.head

    for dep, lemma in reversed(down_path):

        path.append("DOWN:" + dep)

        path.append(lemma)

    return path


# GENERATE SUBSEQUENCES FROM DEPENDENCY PATH

def get_subsequences(path, max_length=3):

    features = Counter()

    for length in range(
        1,
        min(max_length, len(path)) + 1
    ):

        features.update(combinations(path, length))

    return features



def subsequence_kernel(features1, features2):

    common_features = set(features1) & set(features2)

    return float(
        sum(
            features1[f] * features2[f]
            for f in common_features
        )
    )

training_data = [


    ("Steve Jobs founded Apple.",
     "Steve Jobs", "Apple", "FOUNDED"),

    ("Larry Page founded Google.",
     "Larry Page", "Google", "FOUNDED"),

    ("Bill Gates founded Microsoft.",
     "Bill Gates", "Microsoft", "FOUNDED"),

    ("Mark Zuckerberg founded Facebook.",
     "Mark Zuckerberg", "Facebook", "FOUNDED"),

    ("Jeff Bezos founded Amazon.",
     "Jeff Bezos", "Amazon", "FOUNDED"),

    ("Satya Nadella works at Microsoft.",
     "Satya Nadella", "Microsoft", "WORKS_FOR"),

    ("Sundar Pichai works at Google.",
     "Sundar Pichai", "Google", "WORKS_FOR"),

    ("Tim Cook works at Apple.",
     "Tim Cook", "Apple", "WORKS_FOR"),

    ("Sheryl Sandberg works at Facebook.",
     "Sheryl Sandberg", "Facebook", "WORKS_FOR"),

    ("Andy Jassy works at Amazon.",
     "Andy Jassy", "Amazon", "WORKS_FOR"),

    ("Paris is located in France.",
     "Paris", "France", "LOCATED_IN"),

    ("Mumbai is located in India.",
     "Mumbai", "India", "LOCATED_IN"),

    ("Tokyo is located in Japan.",
     "Tokyo", "Japan", "LOCATED_IN"),

    ("Berlin is located in Germany.",
     "Berlin", "Germany", "LOCATED_IN"),

    ("Cairo is located in Egypt.",
     "Cairo", "Egypt", "LOCATED_IN"),

    ("Steve Jobs visited Paris.",
     "Steve Jobs", "Paris", "NONE"),

    ("Larry Page met Bill Gates.",
     "Larry Page", "Bill Gates", "NONE"),

    ("Google opened an office in Mumbai.",
     "Google", "Mumbai", "NONE"),

    ("Apple launched a product in France.",
     "Apple", "France", "NONE"),

    ("Sundar Pichai traveled to Japan.",
     "Sundar Pichai", "Japan", "NONE"),

    ("Microsoft sponsored an event in London.",
     "Microsoft", "London", "NONE")

]


def find_entity(doc, entity_name):

    for ent in doc.ents:

        if ent.text.casefold() == entity_name.casefold():

            return ent

    return None

train_paths = []

train_labels = []

print("\nPreparing training data...\n")

for sentence, entity1_name, entity2_name, relation in training_data:

    doc = nlp(sentence)

    ent1 = find_entity(doc, entity1_name)

    ent2 = find_entity(doc, entity2_name)

    if ent1 is None or ent2 is None:

        print("Skipped:", sentence)

        continue

    path = get_dependency_path(ent1, ent2)

    if path:

        train_paths.append(path)

        train_labels.append(relation)


print("Training examples:", len(train_paths))

print("Relationship classes:", set(train_labels))


required_classes = {
    "FOUNDED",
    "WORKS_FOR",
    "LOCATED_IN",
    "NONE"
}

missing_classes = required_classes - set(train_labels)

if missing_classes:

    raise ValueError(
        "Some relationship classes are missing: "
        + str(missing_classes)
        + ". Check the skipped training examples."
    )

train_features = [

    get_subsequences(path)

    for path in train_paths

]

n = len(train_features)

kernel_matrix = np.zeros((n, n), dtype=float)

for i in range(n):

    for j in range(i, n):

        value = subsequence_kernel(
            train_features[i],
            train_features[j]
        )
        kernel_matrix[i][j] = value

        kernel_matrix[j][i] = value


print("\nKernel Matrix Generated Successfully!")

print("Kernel Matrix Shape:", kernel_matrix.shape)

model = SVC(
    kernel="precomputed",
    C=1.0,
    class_weight="balanced",
    decision_function_shape="ovr"
)

model.fit(kernel_matrix, train_labels)

print("\nSVM Model Trained Successfully!")

print("Classes:", model.classes_)

class_to_index = {

    label: index

    for index, label in enumerate(model.classes_)

}


def predict_direction(ent1, ent2):
    path = get_dependency_path(ent1, ent2)

    if not path:

        return None
    test_features = get_subsequences(path)

    test_kernel = np.array([

        subsequence_kernel(test_features, features)

        for features in train_features

    ], dtype=float).reshape(1, -1)

    scores = model.decision_function(test_kernel).ravel()

    none_index = class_to_index["NONE"]

    relation_indices = [

        i for i, label in enumerate(model.classes_)

        if label != "NONE"

    ]

    best_index = max(
        relation_indices,
        key=lambda i: scores[i]
    )

    best_relation = model.classes_[best_index]

    margin = float(scores[best_index] - scores[none_index])

    return {

        "Entity 1": ent1.text,

        "Entity 2": ent2.text,

        "Path": path,

        "Relation": best_relation,

        "Margin": margin

    }


# eXTRACT RELATIONSHIPS FROM NEW TEXT

def extract_relations(text, include_no_relation=False):

    doc = nlp(text)

    entities = list(doc.ents)

    results = []

    for ent1, ent2 in combinations(entities, 2):

        forward = predict_direction(ent1, ent2)

        backward = predict_direction(ent2, ent1)

        candidates = [

            result

            for result in [forward, backward]

            if result is not None

        ]

        if not candidates:

            continue

        best = max(
            candidates,
            key=lambda result: result["Margin"]
        )

        if best["Margin"] > 0:

            relation = best["Relation"]

            entity1 = best["Entity 1"]

            entity2 = best["Entity 2"]

            path = best["Path"]

        else:

            relation = "NONE"

            entity1 = ent1.text

            entity2 = ent2.text

            path = (
                forward["Path"]
                if forward is not None
                else backward["Path"]
            )

        if relation != "NONE" or include_no_relation:

            results.append({

                "Entity 1": entity1,

                "Entity 2": entity2,

                "Dependency Path": " ".join(path),

                "Relation": relation

            })

    return results


test_sentences = [

    "Steve Jobs founded Pixar.",

    "Tim Cook works at Apple.",

    "London is located in England.",

    "Steve Jobs visited Paris."

]


for text in test_sentences:

    print("\n====================================")

    print("Input Text:", text)

    print("====================================")

    results = extract_relations(
        text,
        include_no_relation=True
    )

    if results:

        for result in results:

            print("\nEntity 1:", result["Entity 1"])

            print("Entity 2:", result["Entity 2"])

            print(
                "Dependency Path:",
                result["Dependency Path"]
            )

            print("Relationship:", result["Relation"])

    else:

        print("No named entity relationships found.")


print("\n\n========== CUSTOM INPUT ==========")

user_text = input("Enter a sentence: ")

results = extract_relations(
    user_text,
    include_no_relation=True
)

print("\nExtracted Relationships:\n")

if results:

    for result in results:

        print(result)

else:

    print("No relationships found.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 124.0 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
SpaCy model loaded successfully!

Preparing training data...

Skipped: Mark Zuckerberg founded Facebook.
Skipped: Sheryl Sandberg works at Facebook.
Training examples: 19
Relationship classes: {'NONE', 'LOCATED_IN', 'FOUNDED', 'WORKS_FOR'}

Kernel Matrix Generated Successfully!
Kernel Matrix Shape: (19, 19)

SVM Model Trained Successfully!
Classes: ['FOUNDED' 'LOCATED_IN' 'NONE' 'WORKS_FOR']

Input Text: Steve Jobs founded Pixar.

Entity 1: Steve Jobs
Entity 2: Pixar
Dependency Path: jobs UP:nsubj found DOWN:dobj pixar
Relationship: FOUNDED

Input Text: Tim Cook works at Apple.
